In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve
from pycaret.anomaly import load_model, predict_model
from src.evaluate import prepare_test_data

In [3]:
df_test = prepare_test_data()

y_true_full = df_test["Ground_Truth"].values

data_for_prediction = df_test.drop(columns = ["Ground_Truth"])

--- Preparando Dados de Teste ---
Carregando Normal...
Memória inicial: 9.89 MB
Memória final: 4.97 MB (49.8% redução)
Carregando Falhas (Isso pode demorar)...
Memória inicial: 197.75 MB
Memória final: 99.33 MB (49.8% redução)
Concatenando...
Dataset de teste final: 504000 linhas.
Distribuição: {1: 399985, 0: 104015}


In [4]:

knn_clf = load_model("../models/knn_model")

Transformation Pipeline and Model Successfully Loaded


In [5]:
predictions = predict_model(knn_clf, data=data_for_prediction)

y_pred = predictions["Anomaly"]
y_scores = predictions["Anomaly_Score"]

## Evaluation

In [6]:
# Métricas
prec = precision_score(y_true_full, y_pred)
rec = recall_score(y_true_full, y_pred)
f1 = f1_score(y_true_full, y_pred)
    
print(f"Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

Precision: 0.9869 | Recall: 0.6881 | F1: 0.8108


### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true_full, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predito (PyCaret)')
plt.ylabel('Real (Ground Truth)')
plt.title(f'Matriz de Confusão: {knn_clf}')
plt.close()

### Precision-Recall Curve

In [7]:
plt.figure(figsize=(10, 8))
    
precision, recall, thresholds = precision_recall_curve(y_true_full, y_scores)
        
# Plota a linha do modelo
plt.plot(recall, precision, label=f'knn', linewidth=2)
        
# --- BÔNUS: Descobrindo o Threshold Mágico para o KNN ---
# Queremos encontrar qual é o threshold para ter pelo menos 90% de Recall
# O array de recall vai diminuindo, vamos achar o valor mais próximo de 0.90
idx = (np.abs(recall - 0.90)).argmin() 

# O array de thresholds é 1 item menor que recall/precision, então ajustamos o índice
idx_thresh = min(idx, len(thresholds) - 1)
best_thresh = thresholds[idx_thresh]

print(f"\n[DICA DE OURO] Para o modelo KNN:")
print(f" -> Se você quiser ~90% de Recall...")
print(f" -> Altere seu código para usar o threshold (limiar) de: {best_thresh:.4f}")
print(f" -> A Precisão cairá de 98% para aprox: {precision[idx]:.4f}")
print("-" * 40)

# Marca esse ponto no gráfico com um 'X' vermelho
plt.plot(recall[idx], precision[idx], 'rX', markersize=10, label='Ponto Ideal KNN (~90% Recall)')

plt.xlabel('Recall (Sensibilidade - Pegar todas as falhas)')
plt.ylabel('Precision (Precisão - Evitar falsos alarmes)')
plt.title('Comparação Precision-Recall - Detecção de Anomalias TEP')
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.close()
print("Gráfico Precision-Recall salvo na pasta results.")


[DICA DE OURO] Para o modelo KNN:
 -> Se você quiser ~90% de Recall...
 -> Altere seu código para usar o threshold (limiar) de: 6.2985
 -> A Precisão cairá de 98% para aprox: 0.8801
----------------------------------------
Gráfico Precision-Recall salvo na pasta results.
